In [ ]:
import os, time, threading, multiprocessing, logging

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")

In [ ]:
%%microblaze base.PMODA

#include <stdint.h>
#include <stdbool.h>
#include "pyprintf.h"
#include "spi.h"

#define DEBUG_MODE      1

#define SUCCESS         0
#define ERR_INVALID    -1
#define ERR_NOTREADY   -2
#define ERR_GENERIC    -3

static char rx_buff;
const char  tx_buff = '\0';
static bool spi_inited = false;

unsigned int DATA_BYTE_SIZE = 1;
unsigned int SCLK_PIN = 0;
unsigned int MISO_PIN = 1;
unsigned int MOSI_PIN = 2;
unsigned int CS_PIN = 3;

spi spi_dev;

// Initialize SPI device (PYNQ will be master)
int spi_init
(
    void
)
{
    if (spi_inited)
    {
        return SUCCESS;
    }

    spi_dev = spi_open(SCLK_PIN, MISO_PIN, MOSI_PIN, CS_PIN);

    // spi_open returns -1 in case of an error
    if (spi_dev < 0)
    {
        #if DEBUG_MODE
        pyprintf("spi_open failed, err %d\n", spi_dev);
        #endif

        return ERR_GENERIC;
    }

    spi_configure(spi_dev, 0, 0); // CPOL = 0, CPHA = 0 (mode 0)

    #if DEBUG_MODE
    pyprintf("spi_open successful");
    #endif

    spi_inited = true;

    return SUCCESS;
}

// Read data over SPI
unsigned int spi_read_data
(
    void
)
{
    if (!spi_inited)
    {
        return ERR_NOTREADY;
    }

    spi_transfer(spi_dev, &tx_buff, &rx_buff, DATA_BYTE_SIZE);
    return (unsigned int)rx_buff;
}

In [ ]:
%%microblaze base.PMODB

#include <stdint.h>
#include <stdbool.h>
#include "pyprintf.h"
#include "xio_switch.h"
#include "gpio.h"
#include "timer.h"

#define DEBUG_MODE      0

#define SUCCESS         0
#define ERR_INVALID    -1
#define ERR_NOTREADY   -2

#define NUM_PINS        8

#define TIMER_FREQ_MHZ  100

typedef struct
{
    bool inited;
    gpio gpio_dev;

} gpio_handle_t;

typedef struct
{
    bool  inited;
    timer timer_dev;

} timer_handle_t;

gpio_handle_t  gpio_handles[NUM_PINS];
timer_handle_t timer_handles[NUM_PINS];

// Initialization function to cache a GPIO handle and set the
// direction; this may be more efficient for repeated IO operations

int init_gpio
(
    unsigned int pin,
    unsigned int direction
)
{
    #if DEBUG_MODE
    pyprintf("init_gpio called for pin %u\n", pin);
    #endif

    if (pin >= NUM_PINS)
    {
        return ERR_INVALID;
    }

    if ((direction != GPIO_IN) && (direction != GPIO_OUT))
    {
        return ERR_INVALID;
    }

    gpio_handles[pin].gpio_dev = gpio_open(pin);
    gpio_set_direction(gpio_handles[pin].gpio_dev, direction);
    gpio_handles[pin].inited = true;

    #if DEBUG_MODE
    pyprintf("init_gpio done for pin %u\n", pin);
    #endif

    return SUCCESS;
}

// Function to write to a PMOD pin

int write_gpio
(
    unsigned int pin,
    unsigned int val
)
{
    if (pin >= NUM_PINS)
    {
        return ERR_INVALID;
    }

    if (!gpio_handles[pin].inited)
    {
        return ERR_NOTREADY;
    }

    if (val > 1) {
        // Technically, an error, but just limit the user input to 1
        val = 1;
    }

    gpio_write(gpio_handles[pin].gpio_dev, val);

    return SUCCESS;
}

// Function to read a PMOD pin

int read_gpio
(
    unsigned int pin
)
{
    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    if (!gpio_handles[pin].inited)
    {
        return ERR_NOTREADY;
    }

    return gpio_read(gpio_handles[pin].gpio_dev);
}

// Initialization function to cache a timer handle and set the
// direction; this may be more efficient for repeated IO operations

int init_pwm
(
    unsigned int pin
)
{
    #if DEBUG_MODE
    pyprintf("init_pwm called for pin %u\n", pin);
    #endif

    if (pin >= NUM_PINS)
    {
        return ERR_INVALID;
    }

    timer_handles[pin].timer_dev = timer_open_device(0);
    init_io_switch();
    set_pin(pin, PWM0);
    timer_handles[pin].inited = true;

    #if DEBUG_MODE
    pyprintf("init_pwm done for pin %u\n", pin);
    #endif

    return SUCCESS;
}

int start_pwm
(
    unsigned int pin,
    unsigned int period_usec,
    unsigned int duty_cycle
)
{
    #if DEBUG_MODE
    pyprintf("start_pwm called for pin %u\n", pin);
    #endif

    if (pin >= NUM_PINS)
    {
        return ERR_INVALID;
    }

    if ((period_usec == 0) || (period_usec >= 65536) ||
        (duty_cycle  == 0) || (duty_cycle >= 100))
    {
        return ERR_INVALID;
    }

    if (!timer_handles[pin].inited)
    {
        return ERR_NOTREADY;
    }

    // Convert the period input in microseconds to AXI Timer clock ticks
    unsigned int period = period_usec * TIMER_FREQ_MHZ;

    // Calculate the pulse from duty_cycle
    unsigned int pulse = duty_cycle * period / 100;

    timer_pwm_generate(timer_handles[pin].timer_dev, period, pulse);

    #if DEBUG_MODE
    pyprintf("start_pwm done for pin %u\n", pin);
    #endif

    return SUCCESS;
}

int stop_pwm
(
    unsigned int pin
)
{
    #if DEBUG_MODE
    pyprintf("stop_pwm called for pin %u\n", pin);
    #endif

    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    timer_pwm_stop(timer_handles[pin].timer_dev);

    #if DEBUG_MODE
    pyprintf("stop_pwm done for pin %u\n", pin);
    #endif

    return SUCCESS;
}

In [ ]:
%%microblaze base.ARDUINO

#include <stdint.h>
#include <stdbool.h>
#include "xio_switch.h"
#include "gpio.h"
#include "timer.h"

// TODO: PWM control does not seem to be working
#define ENABLE_PWM      0

#define SUCCESS         0
#define ERR_INVALID    -1
#define ERR_NOTREADY   -2

#define _BV(bit)   (1 << (bit))

#define NUM_MOTORS        4

// Arduino pin numbers for interface to 74HCT595 latch
#define MOTOR_PIN_LATCH   12
#define MOTOR_PIN_CLK     4
#define MOTOR_PIN_ENABLE  7
#define MOTOR_PIN_DATA    8

// Arduino pin numbers for motor PWM control
#define MOTOR1_PWM_PIN    11
#define MOTOR2_PWM_PIN    3
#define MOTOR3_PWM_PIN    6

// TODO: Pin 5 remains at 0 V if PWM is enabled, causing motor 4
//       (front-right) to not work
#define MOTOR4_PWM_PIN    5

// Bit positions in the 74HCT595 shift register output
#define MOTOR1_A  2
#define MOTOR1_B  3
#define MOTOR2_A  1
#define MOTOR2_B  4
#define MOTOR4_A  0
#define MOTOR4_B  6
#define MOTOR3_A  5
#define MOTOR3_B  7

// Constants that the user passes in to the motor calls
#define FORWARD   1
#define BACKWARD  2
#define BRAKE     3
#define RELEASE   4

#define TIMER_FREQ_MHZ   100
#define PWM_FREQ_KHZ     2
#define PWM_PERIOD_TICKS ((1000 / PWM_FREQ_KHZ) * TIMER_FREQ_MHZ)

static uint8_t latch_state;
static bool    motor_controller_enabled = false;

gpio motor_pin_latch;
gpio motor_pin_enable;
gpio motor_pin_data;
gpio motor_pin_clk;

typedef struct
{
    bool  inited;
    timer timer_dev;

} timer_handle_t;

timer_handle_t timer_handles[NUM_MOTORS];

static void MotorController_latch_tx
(
    void
)
{
    uint8_t i;

    gpio_write(motor_pin_latch, 0);
    gpio_write(motor_pin_data, 0);

    for (i = 0; i < 8; i++)
    {
        gpio_write(motor_pin_clk, 0);

        if (latch_state & _BV(7-i))
        {
            gpio_write(motor_pin_data, 1);
        }
        else
        {
            gpio_write(motor_pin_data, 0);
        }

        gpio_write(motor_pin_clk, 1);
    }

    gpio_write(motor_pin_latch, 1);
}

static void MotorController_enable
(
    void
)
{
    // Only need to enable the motor controller once
    if (motor_controller_enabled)
    {
        return;
    }

    init_io_switch();
    motor_pin_latch  = gpio_open(MOTOR_PIN_LATCH);
    motor_pin_enable = gpio_open(MOTOR_PIN_ENABLE);
    motor_pin_data   = gpio_open(MOTOR_PIN_DATA);
    motor_pin_clk    = gpio_open(MOTOR_PIN_CLK);

    gpio_set_direction(motor_pin_latch, GPIO_OUT);
    gpio_set_direction(motor_pin_enable, GPIO_OUT);
    gpio_set_direction(motor_pin_data, GPIO_OUT);
    gpio_set_direction(motor_pin_clk, GPIO_OUT);

    latch_state = 0;

    MotorController_latch_tx();  // "reset"

    gpio_write(motor_pin_enable, 0);

    motor_controller_enabled = true;
}

// num: Motor number
// speed: 1-100 number
int DCMotor_init
(
    unsigned int motornum,
    unsigned int init_speed
)
{
    unsigned int num, pulse;

    if ((motornum   == 0) || (motornum   > 4) ||
        (init_speed == 0) || (init_speed > 100))
    {
        return ERR_INVALID;
    }

    MotorController_enable();

    // Calculate the pulse from initial speed input (duty cycle)
    pulse = init_speed * PWM_PERIOD_TICKS / 100;

    num = motornum - 1;

    // PWMn for set_pin() is referenced from
    // https://github.com/Xilinx/PYNQ/blob/master/docs/source/overlay_design_methodology/pynq_microblaze_subsystem.rst
    switch (motornum)
    {
        case 1:
            latch_state &= ~_BV(MOTOR1_A) & ~_BV(MOTOR1_B);
            MotorController_latch_tx();

            #if ENABLE_PWM
            timer_handles[num].timer_dev = timer_open(MOTOR1_PWM_PIN);
            set_pin(MOTOR1_PWM_PIN, PWM0);
            timer_handles[num].inited = true;
            timer_pwm_generate(
                timer_handles[num].timer_dev, PWM_PERIOD_TICKS, pulse);
            #endif

            break;

        case 2:
            latch_state &= ~_BV(MOTOR2_A) & ~_BV(MOTOR2_B);
            MotorController_latch_tx();

            #if ENABLE_PWM
            timer_handles[num].timer_dev = timer_open(MOTOR2_PWM_PIN);
            set_pin(MOTOR2_PWM_PIN, PWM1);
            timer_handles[num].inited = true;
            timer_pwm_generate(
                timer_handles[num].timer_dev, PWM_PERIOD_TICKS, pulse);
            #endif

            break;

        case 3:
            latch_state &= ~_BV(MOTOR3_A) & ~_BV(MOTOR3_B);
            MotorController_latch_tx();

            #if ENABLE_PWM
            timer_handles[num].timer_dev = timer_open(MOTOR3_PWM_PIN);
            set_pin(MOTOR3_PWM_PIN, PWM2);
            timer_handles[num].inited = true;
            timer_pwm_generate(
                timer_handles[num].timer_dev, PWM_PERIOD_TICKS, pulse);
            #endif

            break;

        case 4:
            latch_state &= ~_BV(MOTOR4_A) & ~_BV(MOTOR4_B);
            MotorController_latch_tx();

            #if ENABLE_PWM
            timer_handles[num].timer_dev = timer_open(MOTOR4_PWM_PIN);
            set_pin(MOTOR4_PWM_PIN, PWM3);
            timer_handles[num].inited = true;
            timer_pwm_generate(
                timer_handles[num].timer_dev, PWM_PERIOD_TICKS, pulse);
            #endif

            break;

        default:
            return ERR_INVALID;
    }

    return SUCCESS;
}

int DCMotor_run
(
    unsigned int motornum,
    unsigned int cmd
)
{
    uint8_t a, b;

    switch (motornum)
    {
        case 1:
            a = MOTOR1_A; b = MOTOR1_B; break;
        case 2:
            a = MOTOR2_A; b = MOTOR2_B; break;
        case 3:
            a = MOTOR3_A; b = MOTOR3_B; break;
        case 4:
            a = MOTOR4_A; b = MOTOR4_B; break;
        default:
            return ERR_INVALID;
    }

    switch (cmd)
    {
        case FORWARD:
            latch_state |= _BV(a);
            latch_state &= ~_BV(b); 
            MotorController_latch_tx();
            break;

        case BACKWARD:
            latch_state &= ~_BV(a);
            latch_state |= _BV(b); 
            MotorController_latch_tx();
            break;

        case RELEASE:
            latch_state &= ~_BV(a);
            latch_state &= ~_BV(b); 
            MotorController_latch_tx();
            break;
        default:
            return ERR_INVALID;
    }

    return SUCCESS;
}

int DCMotor_setSpeed
(
    unsigned int motornum,
    unsigned int speed
)
{
    unsigned int num, pulse;

    if ((motornum == 0) || (motornum > 4) ||
        (speed    == 0) || (speed    > 100))

    {
        return ERR_INVALID;
    }

    // Calculate the pulse from initial speed input (duty cycle)
    pulse = speed * PWM_PERIOD_TICKS / 100;
    num = motornum - 1;

    #if ENABLE_PWM
    timer_pwm_stop(timer_handles[num].timer_dev);
    timer_pwm_generate(
        timer_handles[num].timer_dev, PWM_PERIOD_TICKS, pulse);
    #endif

    return SUCCESS;
}

In [ ]:
# SPI Helper functions
def get_laser_state(data_byte):
    return ((data_byte >> 7) & 1)

def get_motor_cmd(data_byte):
    return (data_byte & 0xF)

# Constants

# GPIO direction constants (referenced from MicroBlaze gpio.h)
# See https://github.com/Xilinx/PYNQ/blob/master/boards/sw_repo/pynqmb/src/gpio.h
GPIO_OUT = 0
GPIO_IN  = 1

# Following constants were referenced from AdaFruit motor shield v1 library
# See https://github.com/adafruit/Adafruit-Motor-Shield-library

# Constants that the user passes in to the motor calls
FORWARD  = 1
BACKWARD = 2
BRAKE    = 3
RELEASE  = 4

# Constants that encode the joystick D-pad user input
# This corresponds to map_to_direction() in joystick_controller.ino
DPAD_NEUTRAL        = 0
DPAD_LEFT           = 1
DPAD_RIGHT          = 2
DPAD_FORWARD        = 3
DPAD_BACKWARD       = 4
DPAD_FORWARD_LEFT   = 5
DPAD_FORWARD_RIGHT  = 6
DPAD_BACKWARD_LEFT  = 7
DPAD_BACKWARD_RIGHT = 8

cmd_map = [
    "neutral",
    "left",
    "right",
    "forward",
    "backward",
    "forward_left",
    "forward_right",
    "backward_left",
    "backward_right",
]

In [ ]:
class DCMotor:
    def __init__(self, motornum, speed = 20):
        self.motornum = motornum
        self.speed = speed

        DCMotor_init(motornum, speed)

        #self.set_speed_event = threading.Event()
        #self.set_speed_thread = threading.Thread(
        #    target = self.set_speed_t,
        #    args = ())

        #self.set_speed_thread.start()

    #def set_speed_t(self):
    #    print(f"Set speed thread for motor {self.motornum} started")

    def run(self, cmd):
        # TODO: For testing, skip front motors because, *meow*
        if (self.motornum == 1) or (self.motornum == 4):
            return

        DCMotor_run(self.motornum, cmd)

    def set_speed(self, speed):
        DCMotor_setSpeed(self.motornum, speed)


In [ ]:
class RC_Car:
    def __init__(self, weapons = True, log_level = logging.INFO):
        # Logging
        self.logger = logging.getLogger("PYNQ-Tag")
        self.logger.setLevel(log_level)
        self.logfile_handler = logging.FileHandler("PYNQ-Tag.log", mode="w")

        self.logfile_handler.setFormatter(
            logging.Formatter(
                "%(asctime)s.%(msecs)03d - %(levelname)s - %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S"))
        self.logger.addHandler(self.logfile_handler)

        self.logger.info(f"Beginning program...")
        self.logger.debug(f"Logging initialized")

        # Joystick link
        err = spi_init()
        if (err != 0):
            self.logger.error(
                f"spi_init failed, err: {err}")
            return
        else:
            self.logger.debug(
                f"spi_init successful")

        # Motors
        self.logger.debug(f"Initializing motors")
        # TODO: Config file?
        # Motor 0 is front-left (M1)
        # Motor 1 is back-left (M2)
        # Motor 2 is back-right (M3)
        # Motor 3 is front-right (M4)
        self.motors = []
        for i in range(1, 5):
            self.motors.append(DCMotor(i))
        self.motor_fl = self.motors[0]
        self.motor_fr = self.motors[3]
        self.motor_bl = self.motors[1]
        self.motor_br = self.motors[2]
        self.logger.debug(f"Motors initialized")

        # Weapons
        self.weapons = weapons
        if weapons:
            self.logger.debug(f"Initializing laser and IR receiver")
        else:
            self.logger.debug(f"Skipping laser and IR receiver initialization")

        self.pmodB_lock = threading.Lock()
        self.laser = self.Laser(parent_class = self)
        self.ir_receiver = self.IR_Receiver(parent_class = self)
        if weapons:
            self.logger.debug(f"Laser and IR receiver initialized")

    def fire_laser(self):
        self.laser.shoot_event.set()

    def stop(self):
        self.logger.info(f"Stopping program...")
        self.logfile_handler.close()
        for motor in self.motors:
            motor.run(RELEASE)

    def move(self, cmd):
        global cmd_map
        self.logger.debug(f"Received command: {cmd_map[cmd]}")
        if (cmd == DPAD_FORWARD):
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_BACKWARD):
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_LEFT):
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_RIGHT):
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_FORWARD_LEFT):
            #self.motor_fl.set_speed(20)
            #self.motor_bl.set_speed(20)
            self.motor_fl.run(RELEASE)
            #self.motor_fl.run(FORWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(RELEASE)
            #self.motor_bl.run(FORWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_FORWARD_RIGHT):
            #self.motor_fr.set_speed(20)
            #self.motor_br.set_speed(20)
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(RELEASE)
            #self.motor_fr.run(FORWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(RELEASE)
            #self.motor_br.run(FORWARD)

        elif (cmd == DPAD_BACKWARD_LEFT):
            #self.motor_fl.set_speed(20)
            #self.motor_bl.set_speed(20)
            self.motor_fl.run(RELEASE)
            #self.motor_fl.run(BACKWARD)
            self.motor_fr.run(BACKWARD)
            #self.motor_bl.run(BACKWARD)
            self.motor_bl.run(RELEASE)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_BACKWARD_RIGHT):
            #self.motor_fr.set_speed(20)
            #self.motor_br.set_speed(20)
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(RELEASE)
            #self.motor_fr.run(BACKWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(RELEASE)
            #self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_NEUTRAL):
            self.motor_fl.run(RELEASE)
            self.motor_fr.run(RELEASE)
            self.motor_bl.run(RELEASE)
            self.motor_br.run(RELEASE)

    class Laser:
        def __init__(self, parent_class, pin = 7):
            # IR transmitter "laser" is on PMODB and connected to
            # pin 7

            self.logger = parent_class.logger
            self.enable = parent_class.weapons
            if not self.enable:
                self.logger.info(f"Laser not initialized")
                return

            err = init_pwm(pin)
            if (err != 0):
                self.logger.error(
                    f"init_pwm failed for pin {pin}, err: {err}")
            else:
                self.logger.debug(
                    f"init_pwm successful for pin {pin}")

            self.pwm_period_usec = 26 # 26 usec is about 38.4 KHz
            self.pwm_duty_cycle = 50
            self.pwm_pin = pin
            self.laser_pulse_duration_msec = 250 # 250 msec pulse

            self.shoot_event = threading.Event()
            self.shoot_thread = threading.Thread(
                target = self.shoot_t,
                args = (parent_class.pmodB_lock, self.shoot_event))

            self.shoot_thread.start()

            self.logger.info(f"Initialized laser shoot thread")
            self.logger.debug(f"Laser signal pin: {self.pwm_pin}")
            self.logger.debug(
                f"PWM frequency: {1000.0 / self.pwm_period_usec:.2f} KHz")
            self.logger.debug(
                f"Laser pulse duration: {self.laser_pulse_duration_msec} milliseconds")

        def shoot_t(self, lock, shoot_event):
            while True:
                # Wait until shoot_event is set
                if not shoot_event.is_set():
                    continue

                self._shoot(lock)
                shoot_event.clear()

        def _shoot(self, lock):
            if not self.enable:
                self.logger.debug(f"Laser could not be shot - not initialized!")
                return

            # Shoot a quarter second pulse "laser"
            lock.acquire()
            err = start_pwm(
                self.pwm_pin,
                self.pwm_period_usec,
                self.pwm_duty_cycle)
            lock.release()

            if (err != 0):
                self.logger.error(
                    f"start_pwm failed for pin {self.pwm_pin}, err: {err}")
            else:
                self.logger.debug(
                    f"start_pwm successful for pin {self.pwm_pin}")

            time.sleep(self.laser_pulse_duration_msec / 1000)

            lock.acquire()
            err = stop_pwm(self.pwm_pin)
            lock.release()

            if (err != 0):
                self.logger.error(
                    f"stop_pwm failed for pin {self.pwm_pin}, err: {err}")
            else:
                self.logger.debug(
                    f"stop_pwm successful for pin {self.pwm_pin}")

            self.logger.info(f"Laser shot!")

    class IR_Receiver():
        def __init__(self, parent_class, pin = 3):
            # IR receiver is on PMODB and connected to pin 3
            self.enable = parent_class.weapons
            self.logger = parent_class.logger
            if not self.enable:
                self.logger.info(f"IR receiver not initialized")
                return

            err = init_gpio(pin, GPIO_IN)
            if (err != 0):
                self.logger.error(
                    f"init_gpio failed for pin {pin}, err: {err}")
            else:
                self.logger.debug(
                    f"init_gpio successful for pin {pin}")

            self.ir_pin = pin

            # Start a process which indicates a hit
            self.ir_rx_thread = threading.Thread(
                target = self.notify_hit_t,
                args = (parent_class.pmodB_lock, ))

            self.ir_rx_thread.start()

            self.logger.info(f"Initialized IR Receiver thread")
            self.logger.debug(f"IR receiver signal pin: {self.ir_pin}")

        def notify_hit_t(self, lock):
            if not self.enable:
                return

            lock.acquire()
            prev_read = read_gpio(self.ir_pin)
            lock.release()
            while True:
                lock.acquire()
                curr_read = read_gpio(self.ir_pin)
                lock.release()
                if curr_read == 0 and curr_read != prev_read:
                    # TODO: This will send a kill signal?
                    self.logger.info("Hit!")
                    print("Hit!")
                prev_read = curr_read
                time.sleep(0.01)


In [ ]:
car = RC_Car(True, logging.DEBUG)

In [ ]:
prev_cmd = -1

while True:
    data = spi_read_data()
    laser = get_laser_state(data)
    cmd = get_motor_cmd(data)

    if cmd >= DPAD_NEUTRAL and cmd <= DPAD_BACKWARD_RIGHT:
        if cmd != prev_cmd:
            print(f"Direction command received: {cmd}")
        car.move(cmd)
        prev_cmd = cmd

    if laser == 1:
        print(f"Laser fired: {laser}")
        car.fire_laser()

    time.sleep(0.01)